In [2]:
from langchain_community.document_loaders import PyPDFLoader  
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
from langchain_openai import OpenAIEmbeddings 
from dotenv import load_dotenv 
import os

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI 

from langchain_classic.memory import ChatMessageHistory

from langchain_core.output_parsers import StrOutputParser

In [4]:
loader = PyPDFLoader('data/2040_seoul_plan.pdf')
data_seoul = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
seoul_splits = text_splitter.split_documents(data_seoul)
for i in range(len(seoul_splits) - 1):
    seoul_splits[i].page_content += '\n' + seoul_splits[i + 1].page_content[:100]
for split in seoul_splits[:5]:
    print('-------------------------------')
    print(split)    

-------------------------------
page_content='「2040 서울도시기본계획」을 발간하며
지난 3년간 코로나19 팬데믹으로 전 세계가 심각한 타격을 받아왔지만, 대한민국의 수도 서울은 혁신적인 디지털 기술과 뛰어난 시민 의식, 풍부한 자연환경을 토대로 도시의 가능성과 잠재력을 확인할 수 있었습니다. 「2040 서울도시기본계획」은 기후위기, 디지털 전환, 생활양식의 변화 등 글로벌 대도시가 당면한 과제에 대한 해법을 제시하고 있습니다.첫째, 보행일상권으로의 공간구조 개편입니다. 서울을 하나의 기준으로 관리하던 지금까지의 방식에서 벗어나, 미래의 서울은 다양한 지역특성을 반영한 차별화된 계획체계를 통해 동네단위의 자족적 생활권으로 재구성하게 됩니다. 이는 기후위기, 팬데믹 등 각종 재난상황에서도 도시활동이 가능한 공간단위로, 서울 어디에서나 시민 삶의 질이 보장되는 새로운 차원의 균형발전정책 기반이 될 것입니다.둘째, 과감하고 유연한 도시계획 기조로의 전환입니다. 앞으로의 도시계획은 미래 여건변화에 유연하고 신속하게 대응할 수 있는 체계를 통해 현장에서 강력하게 작동하는 수단이 될 것입니다. 미래지향적인 계획철학을 토대로 융복합적인 토지이용제도를 실현하고 수변녹지와 연계된 생활공간을 조성하는 등 도시계획체계를 과감하고 유연하게 전환했습니다.
셋째, 미래 변화 속 지속가능한 도시의 구축입니다. 디지털 대전환과 기후위기 등 급변하는 시대에 발맞춰 지속가능한 첨단 도시를 구축하기 위한 구체적인 목표를 수립했습니다. 미래 신' metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2020 11.0.0.5178', 'creationdate': '2024-12-12T18:16:11+09:00', 'author': 'SI', 'moddate': '2024-12-12T18:16:11+09:00', 'pdfversion': '1.4', 'source': 'data/2040_seoul_plan

In [5]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [6]:
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large', api_key=api_key)
v = embedding.embed_query('서울시의 환경정책을 알려줘')
print(v)

[0.013040975667536259, -0.0799129456281662, -0.0075422548688948154, -0.014237518422305584, 0.007004482671618462, 0.031056344509124756, 0.004470231477171183, 0.009249681606888771, -0.039391811937093735, -0.002006562426686287, -0.01462740357965231, -0.0013503123773261905, 0.017975036054849625, 0.05684252083301544, 0.026256727054715157, -0.024885408580303192, -0.029120365157723427, -0.014990399591624737, 0.04380154609680176, -0.01125288289040327, 0.025436624884605408, -0.03291165828704834, 0.006638125516474247, 0.015447506681084633, -0.033798981457948685, 0.007777530234307051, 0.03629962354898453, 0.02042189985513687, 0.017491040751338005, 0.021309223026037216, 0.04025224968791008, 0.03175544738769531, 0.012798978015780449, 0.023514090105891228, -0.030115243047475815, 0.01214020699262619, 0.05619719251990318, -0.007407811935991049, -0.0003413592930883169, 0.03726761415600777, -0.03866581991314888, -0.002332586795091629, 0.015622282400727272, -0.023097315803170204, -0.004510564263910055, 0

In [7]:
persist_directory = 'chroma_store'
if not os.path.exists(persist_directory):
    print('creating...')
    vectorstore = Chroma.from_documents(
        documents=seoul_splits, 
        embedding=embedding, 
        persist_directory=persist_directory
    )
else:
    print('loading...')
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding
    )

loading...


In [8]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke('서울시의 환경정책을 알려줘')
for d in docs:
    print(d)
    print('------------')

page_content='제4절 기후·환경 부문1. 개요Ÿ기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측면과 연관되어 있어 향후 서울시의 적극적인 대응이 필요하다.Ÿ탄소중립 목표뿐만 아니라 미세먼지로부터 시민 건강을 지키기 위해서는 건물, 교통, 에너지 등 도시의 주요 인프라 전반의 혁신이 요구되며, 이를 위해 새로운 기술과 혁신적 제도가 필요하다. 제로에너지 건물, 친환경 차량 및 교통 인프라의 확대, 자원·에너지 순환 기반 조성으로 온실가스와 미세먼지 배출량을 획기적으로 감축해야 한다. Ÿ기후변화에 따른 폭염, 풍수해, 도심열섬현상 등 기후재난 및 극한 기후현상이 심해질 것으로 전망되어 보다 능동적인 대비가 필요하다. Ÿ한편, 환경보존과 쾌적한 도시환경을 위해 도심 곳곳 시민 모두가 누릴 수 있는 도심숲과 생활공원 등 녹색공간을 조성하고, 이를 수변 공간과 연계하여 풍부하고 지속가능한 자연환경이 확보될 수 있도록 한다.Ÿ장기적인 측면에서 시민 개개인과 기업 등 다양한 도시 내 행위자의 적극적인 협조가 필수적이며 이를 위해 중앙정부와 서울시 환경계획 담당부서와의 협력적이고 포용적인 거버넌스 체계를 구축하도록 한다.목표 전략3-12050 탄소중립 실현을 위한 도시 인프라 전환3-1-1건물 부문의 탄소배출을 감축하기 위한 친환경 기술 개발 및 적극 적용3-1-2미래 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라 확충3-1-3에너지 전환을 위한 청정에너지 기반 구축3-1-4대기 환경을 고려한 공간계획과 배출원 관리체계 강화3-2건강한 순환도시 조성을 위한자립적인 자원순환 체계 구축3-2-1자원순환·관리 자립을 위한 분산형 폐기물처리 시설 구축3-2-2기후 행동 포용적 거버넌스 구축을 위한 시민 행동 활성화3-3사람과 자연의 공존을 위한친환경 생태도시 구축3-3-1건물 에너지 분야 효율성 개선 및 도심 속 생물 다양성 확보3-3-2지속가능한 통합 물순환 체계 구축3-4다양한 수변을 경험할 수 있는수변감성도

In [9]:
chat = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '사용자의 질문에 대해 아래 context에 기반하여 답변하라 : \n\n{context}'), 
        MessagesPlaceholder(variable_name='messages')
    ]
)
document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

In [10]:
chat_history = ChatMessageHistory()
chat_history.add_user_message('서울시의 온실가스 저감 정책에 대해 알려줘')
answer = document_chain.invoke(
    {'messages': chat_history.messages, 'context': docs}
)
chat_history.add_ai_message(answer) 
answer

'서울시는 온실가스 저감을 위해 다양한 정책과 전략을 추진하고 있습니다. 주요 내용은 다음과 같습니다:\n\n1. **친환경 기술 개발 및 적용**: 건물 부문에서의 탄소배출을 감축하기 위해 친환경 기술을 개발하고 적극적으로 적용하는 정책이 시행되고 있습니다. 이를 통해 제로에너지 건물의 확대를 목표로 하고 있습니다.\n\n2. **미래 모빌리티와 친환경 차량**: 미래 모빌리티 기술을 활용하고 친환경 수송 차량 및 관련 인프라를 확충하여 대중교통과 개인의 이동 수단에서 발생하는 온실가스를 줄이는 방향으로 나아가고 있습니다.\n\n3. **청정에너지 기반 구축**: 에너지 전환을 위해 청정에너지 기반을 마련하여 화석연료 의존도를 줄이고, 에너지의 효율적 사용을 촉진합니다.\n\n4. **자원순환 체계 구축**: 순환경제를 위한 자원 재활용 및 관리 체계를 강화하고, 지역 내에서 발생하는 자원을 최대한 활용하여 폐기물을 줄이는 전략이 추진됩니다.\n\n5. **대기환경 고려한 공간계획**: 대기 오염물질의 배출을 원천적으로 감축하기 위해 대기환경을 고려한 공간계획을 수립하고, 배출원별 · 계절별 특성에 맞는 맞춤형 대책을 추진합니다.\n\n6. **시민 참여 유도**: 시민의 자원순환 인식 향상을 위한 프로그램과 캠페인을 실시하며, 기후 행동을 실천할 수 있는 다양한 가이드라인을 제공합니다.\n\n7. **협력적 거버넌스 구축**: 중앙정부와 서울시 간의 협력을 강화하여 기후환경 관리 체계를 마련하고, 시민과 기업의 참여를 통해 탄소중립 달성을 위한 포용적인 거버넌스 체계를 구축하고 있습니다.\n\n위와 같은 종합적인 접근을 통해 서울시는 "2050 탄소중립" 목표를 달성하기 위해 지속적으로 노력하고 있습니다.'

In [11]:
query_other = '사회 문화 정책은 어때?'
query_augmentation_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name='messages'),
        ('system', '기존의 대화 내용을 활용하여 사용자가 질문한 의도를 파악해서 한 문장의 명료한 질문으로 변환하라. 대명사나 이, 저, 그와 같은 표현을 명확한 명사로 표현하라. : \n\n{query}')
    ]
)
query_augmentation_chain = query_augmentation_prompt | chat | StrOutputParser()

In [12]:
augmented_query = query_augmentation_chain.invoke({
    'messages': chat_history.messages, 
    'query': query_other
})

In [13]:
docs = retriever.invoke(augmented_query)
for d in docs:
    print(d)
    print('--------------')

page_content='보도 확폭 등으로 도로 공간을 재편하여 보행환경을 개선한다.Ÿ오염물질 배출차량의 통행을 제한하는 녹색교통진흥지역의 적용 공간범위를 확대한다.Ÿ친환경 중·단거리 교통수단인 개인·공공자전거를 비롯하여 PM 등의 이용 증가에 대비하여 시민 편의와 안전을 확보하기 위한 도로 및 이용환경을 조성한다.Ÿ공동이용시설과 대중교통의 미세먼지 배출량 저감을 위한 노력을 적극적으로 전개하여 국제 권고 기준 농도를 달성한다.
제7절 사회·문화 부문95
제7절 사회·문화 부문1. 개요Ÿ서울시민 누구나 성별·연령·지역·인종·국적 등 개인을 구별하는 특성과 무관하게 차별받지 않고 평등한 도시생활을 영위할 수' metadata={'page_label': '102', 'pdfversion': '1.4', 'producer': 'Hancom PDF 1.3.0.542', 'total_pages': 205, 'moddate': '2024-12-12T18:16:11+09:00', 'source': 'data/2040_seoul_plan.pdf', 'author': 'SI', 'creator': 'Hwp 2020 11.0.0.5178', 'creationdate': '2024-12-12T18:16:11+09:00', 'page': 101}
--------------
page_content='제7절 사회·문화 부문95
제7절 사회·문화 부문1. 개요Ÿ서울시민 누구나 성별·연령·지역·인종·국적 등 개인을 구별하는 특성과 무관하게 차별받지 않고 평등한 도시생활을 영위할 수 있도록 교육·의료·복지·문화 등 다양한 방면에서 보호·보장받으며 누릴 수 있는 권익과 가치를 바탕으로 시민생활을 지원한다.Ÿ이러한 도시환경이 보행일상권 내에 균등하게 구축될 수 있도록 지역사회 및 생활권 단위 중심의 생활문화 지원체계를 마련하며, 지역 주민의 사회적 관계형성 사업을 통해 공동체 활동을 확대한다.Ÿ생애발달단계에서 필요한 교육, 취업, 의료 등에 대한 적합한 설계를 통해 시민의 요구를 충족시키고, 

In [14]:
chat_history.add_user_message(query_other)
answer = document_chain.invoke({
    'messages': chat_history.messages, 
    'context': docs
})

In [16]:
print(answer)

서울시의 사회문화 정책은 시민 모두가 차별받지 않고 평등한 도시생활을 영위할 수 있도록 다양한 방면에서 권익과 가치를 보호하고 보장하는 것을 목표로 하고 있습니다. 주요 내용은 다음과 같습니다:

1. **지역사회 지원체계 구축**: 지역 주민의 사회적 관계 형성을 통해 공동체 활동을 확대하고, 균등한 도시환경이 보행일상권 내에 구축될 수 있도록 지역사회 및 생활권 단위 중심의 생활문화 지원체계를 마련합니다.

2. **교육 및 취업 지원**: 생애발달단계에 필요한 교육과 취업 지원 시스템을 구축하여 시민의 요구를 충족시키고 역량을 발휘할 수 있도록 지원합니다. 

3. **차별 없는 생활환경 조성**: 성별, 연령, 지역, 인종, 국적 등 개인의 특성을 구별하지 않고 모두에게 평등하고 건강한 노동환경을 조성하며 적극적인 사회참여 활동을 보장합니다.

4. **사회적 다양성 존중**: 다양한 사회 구성원에 대한 존중과 인정을 바탕으로 문화다양성을 증진하며, 1인 가구, 비혼 가족, 이주민 가족 등 다양한 가족 형태를 인정하고 지원합니다.

5. **문화예술 교육 체계 정착**: 생애주기별 문화예술 교육 체계를 정착시켜 유아에서 노령인구까지 맞춤형 지원을 제공하고, 연령대 간의 교류와 상호작용을 촉진합니다.

6. **고령사회 대응**: 세대 간의 통합과 사회적 돌봄을 강화하기 위한 제도적 지원을 마련하며, 고령자와 청년이 모두 함께 생활할 수 있는 환경을 조성합니다.

7. **사회참여 확대**: 청년 니트(NEET), 은퇴자, 외국인 등을 포함한 다양한 사회적 배경을 가진 시민들이 지속적으로 사회참여 할 수 있도록 관련 활동을 연계하고 지원합니다.

이러한 다양한 정책을 통해 서울시는 시민의 권리와 삶의 질을 향상시켜 모두가 어우러져 살 수 있는 사회문화적 환경을 조성하려고 노력하고 있습니다.
